# Ring oscillator: build once, export to every format

A hierarchical CMOS ring oscillator built with the `kfnetlist.morphnet` builder
API, then written as SPICE, HSPICE, Xyce, Spectre, Verilog-AMS and YAML, and
parsed back to show the conversion is bidirectional.

Adapted from morphnet's `examples/inverter.py`. The changes:

- The `nfet`/`pfet` external modules declare their ports, so positional
  formats (SPICE, Spectre) know the pin order.
- Nets that reach a circuit terminal are named after it. A subcircuit port is
  wired to the node with the same name.
- Instance names follow SPICE's first-letter convention: `X` marks a
  subcircuit or PDK-cell call. Without it, `inv1` would read as a current
  source.

In [1]:
from kfnetlist import morphnet
from kfnetlist.morphnet.netlist import (
    parse_hspice,
    parse_spectre,
    parse_spice,
    parse_vams,
    parse_xyce,
    write_hspice,
    write_spectre,
    write_spice,
    write_vams,
    write_xyce,
)

## Leaf inverter cell

`circuit()` declares a circuit and its parameters (names only, or
`(name, default)` tuples).
`add_terminal()` adds the circuit's I/O, `subcircuit()` places
sub-instances with parameter overrides as keyword arguments, and `connect()`
names a net and lists its endpoints. `("", terminal)` is the circuit's own
terminal.

In [2]:
inv = morphnet.circuit("inverter", parameters=["l", "w"])
inv.add_terminal("vdd", direction=morphnet.PortDirection.INPUT)
inv.add_terminal("gnd", direction=morphnet.PortDirection.INPUT)
inv.add_terminal("inp", direction=morphnet.PortDirection.INPUT)
inv.add_terminal("out", direction=morphnet.PortDirection.OUTPUT)

inv.add(
    morphnet.subcircuit("XMN", "nfet", l=100e-9, w=500e-9),
    morphnet.subcircuit("XMP", "pfet", l=100e-9, w=1000e-9),
)

inv.connect("out", ("XMN", "drain"), ("XMP", "drain"), ("", "out"))
inv.connect("inp", ("", "inp"), ("XMN", "gate"), ("XMP", "gate"))
inv.connect("vdd", ("", "vdd"), ("XMP", "source"))
inv.connect("gnd", ("", "gnd"), ("XMN", "source"))

Circuit(name='inverter', class_name='', ports=[Port(name='vdd', direction=<PortDirection.INPUT: 1>, domain=<SignalDomain.UNSPECIFIED: 0>, width=0, properties={}, discipline=''), Port(name='gnd', direction=<PortDirection.INPUT: 1>, domain=<SignalDomain.UNSPECIFIED: 0>, width=0, properties={}, discipline=''), Port(name='inp', direction=<PortDirection.INPUT: 1>, domain=<SignalDomain.UNSPECIFIED: 0>, width=0, properties={}, discipline=''), Port(name='out', direction=<PortDirection.OUTPUT: 2>, domain=<SignalDomain.UNSPECIFIED: 0>, width=0, properties={}, discipline='')], parameters=[Parameter(name='l', default_value=None, description='', properties={}), Parameter(name='w', default_value=None, description='', properties={})], model_interfaces=[], subcircuits=[SubCircuit(name='XMN', circuit_name='nfet', class_name='', parameters=[Parameter(name='l', default_value=ParameterValue(prefixed_value=PrefixedValue(double_value=1e-07, prefix=<SIPrefix.UNSPECIFIED: 0>), model_ref=None), description='',

## Ring oscillator: three inverters in a loop

One instance overrides `w` through the `parameters` list; the others use the
defaults.

In [3]:
ring = morphnet.circuit("ring_osc")
ring.add_terminal("vdd", direction=morphnet.PortDirection.INPUT)
ring.add_terminal("gnd", direction=morphnet.PortDirection.INPUT)
ring.add_terminal("osc_out", direction=morphnet.PortDirection.OUTPUT)

ring.add(
    morphnet.subcircuit("Xinv1", "inverter", parameters=[("w", 600e-9)]),
    morphnet.subcircuit("Xinv2", "inverter"),
    morphnet.subcircuit("Xinv3", "inverter"),
)
ring.connect("n1", ("Xinv1", "out"), ("Xinv2", "inp"))
ring.connect("n2", ("Xinv2", "out"), ("Xinv3", "inp"))
ring.connect("osc_out", ("Xinv3", "out"), ("Xinv1", "inp"), ("", "osc_out"))
ring.connect("vdd", ("", "vdd"), ("Xinv1", "vdd"), ("Xinv2", "vdd"), ("Xinv3", "vdd"))
ring.connect("gnd", ("", "gnd"), ("Xinv1", "gnd"), ("Xinv2", "gnd"), ("Xinv3", "gnd"))

Circuit(name='ring_osc', class_name='', ports=[Port(name='vdd', direction=<PortDirection.INPUT: 1>, domain=<SignalDomain.UNSPECIFIED: 0>, width=0, properties={}, discipline=''), Port(name='gnd', direction=<PortDirection.INPUT: 1>, domain=<SignalDomain.UNSPECIFIED: 0>, width=0, properties={}, discipline=''), Port(name='osc_out', direction=<PortDirection.OUTPUT: 2>, domain=<SignalDomain.UNSPECIFIED: 0>, width=0, properties={}, discipline='')], parameters=[], model_interfaces=[], subcircuits=[SubCircuit(name='Xinv1', circuit_name='inverter', class_name='', parameters=[Parameter(name='w', default_value=ParameterValue(prefixed_value=PrefixedValue(double_value=6e-07, prefix=<SIPrefix.UNSPECIFIED: 0>), model_ref=None), description='', properties={})], properties={}), SubCircuit(name='Xinv2', circuit_name='inverter', class_name='', parameters=[], properties={}), SubCircuit(name='Xinv3', circuit_name='inverter', class_name='', parameters=[], properties={})], connections=[Connection(name='n1', s

## Netlist: PDK library, device models, circuits

`external_library()` points at the model file, and `ext_module()` declares the
devices it provides, including their pin order.

In [4]:
fet_ports = [morphnet.port(name) for name in ("drain", "gate", "source")]

ckt = morphnet.Netlist(name="ring_osc_top", top_circuit="ring_osc")
ckt.add_library(
    "tt",
    morphnet.external_library(
        "path/to/tt.lib", model_type="tt", external_model_names=["nfet", "pfet"]
    ),
)
ckt.add_ext(
    morphnet.ext_module("nfet", class_name="nmos", library_name="tt", ports=fet_ports),
    morphnet.ext_module("pfet", class_name="pmos", library_name="tt", ports=fet_ports),
)
ckt.add(ring, inv)

assert ckt.top_circuit == "ring_osc"
assert [m.name for m in ckt.circuits] == ["ring_osc", "inverter"]
assert ckt.libraries["tt"].path == "path/to/tt.lib"

## Export to every format

In [5]:
WRITERS = {
    "SPICE": write_spice,
    "HSPICE": write_hspice,
    "Xyce": write_xyce,
    "Spectre": write_spectre,
    "Verilog-AMS": write_vams,
}
texts = {fmt: write(ckt) for fmt, write in WRITERS.items()}
for fmt, text in texts.items():
    print(f"── {fmt} ──\n{text}")

── SPICE ──
* ring_osc_top
.subckt ring_osc vdd gnd osc_out
Xinv1 vdd gnd osc_out n1 inverter w=6e-07
Xinv2 vdd gnd n1 n2 inverter
Xinv3 vdd gnd n2 osc_out inverter
.ends ring_osc
.subckt inverter vdd gnd inp out
XMN out inp gnd nfet l=1e-07 w=5e-07
XMP out inp vdd pfet l=1e-07 w=1e-06
.ends inverter
.end

── HSPICE ──
* ring_osc_top
.subckt ring_osc vdd gnd osc_out
Xinv1 vdd gnd osc_out n1 inverter w=6e-07
Xinv2 vdd gnd n1 n2 inverter
Xinv3 vdd gnd n2 osc_out inverter
.ends ring_osc
.subckt inverter vdd gnd inp out
XMN out inp gnd nfet l=1e-07 w=5e-07
XMP out inp vdd pfet l=1e-07 w=1e-06
.ends inverter
.end

── Xyce ──
* ring_osc_top
.subckt ring_osc vdd gnd osc_out
Xinv1 vdd gnd osc_out n1 inverter w=6e-07
Xinv2 vdd gnd n1 n2 inverter
Xinv3 vdd gnd n2 osc_out inverter
.ends ring_osc
.subckt inverter vdd gnd inp out
XMN out inp gnd nfet l=1e-07 w=5e-07
XMP out inp vdd pfet l=1e-07 w=1e-06
.ends inverter
.end

── Spectre ──
subckt ring_osc (vdd gnd osc_out)
Xinv1 (vdd gnd osc_out n1) i

The same `Netlist` also serializes to YAML (and to JSON or protobuf bytes) as a
portable interchange format:

In [6]:
print(ckt.to_yaml())

directives: []
ext_modules:
- class_name: nmos
  group_name: ''
  kind: 0
  library_name: tt
  name: nfet
  parameters: []
  properties: {}
  terminals:
  - direction: 0
    discipline: ''
    domain: 0
    name: drain
    properties: {}
    width: 0
  - direction: 0
    discipline: ''
    domain: 0
    name: gate
    properties: {}
    width: 0
  - direction: 0
    discipline: ''
    domain: 0
    name: source
    properties: {}
    width: 0
- class_name: pmos
  group_name: ''
  kind: 0
  library_name: tt
  name: pfet
  parameters: []
  properties: {}
  terminals:
  - direction: 0
    discipline: ''
    domain: 0
    name: drain
    properties: {}
    width: 0
  - direction: 0
    discipline: ''
    domain: 0
    name: gate
    properties: {}
    width: 0
  - direction: 0
    discipline: ''
    domain: 0
    name: source
    properties: {}
    width: 0
libraries:
  tt:
    external_model_names:
    - nfet
    - pfet
    model_type: tt
    path: path/to/tt.lib
modules:
- buses: []
  cl

## Parse it back

Every format parses back into a morphnet `Netlist`. Converting each one to
`kfnetlist.Netlist` graphs shows that the connectivity survived every round trip.

Positional formats (the SPICE family and Spectre) record pin *order*, not pin
names, so their instance pins come back as `p0`, `p1`, and so on. Only
Verilog-AMS keeps the names. The check below therefore compares pins by
position.

In [7]:
from kfnetlist import NetlistPort  # noqa: E402

PARSERS = {
    "SPICE": parse_spice,
    "HSPICE": parse_hspice,
    "Xyce": parse_xyce,
    "Spectre": parse_spectre,
    "Verilog-AMS": parse_vams,
}
pin_order = {
    m.name: [p.name for p in m.ports] for m in [*ckt.circuits, *ckt.ext_modules]
}
cell_of = {r.name: r.circuit_name for m in ckt.circuits for r in m.subcircuits}


def pin(member) -> str:
    if isinstance(member, NetlistPort):
        return f"<{member.name}>"
    pins = pin_order[cell_of[member.instance]]
    index = pins.index(member.port) if member.port in pins else int(member.port[1:])
    return f"{member.instance}[{index}]"


def connectivity(netlist: morphnet.Netlist) -> dict[str, list[list[str]]]:
    return {
        name: sorted(sorted(map(pin, net)) for net in nl.nets)
        for name, nl in morphnet.to_netlists(netlist).items()
    }


reference = connectivity(ckt)
for fmt, parse in PARSERS.items():
    assert connectivity(parse(texts[fmt])) == reference, fmt
    print(f"{fmt}: connectivity preserved")

SPICE: connectivity preserved
HSPICE: connectivity preserved
Xyce: connectivity preserved
Spectre: connectivity preserved
Verilog-AMS: connectivity preserved


## Convert between formats

Conversion is a parse followed by a write, for example Spectre → Verilog-AMS:

In [8]:
print(write_vams(parse_spectre(texts["Spectre"])))

module ring_osc (vdd, gnd, osc_out);
  inout vdd, gnd, osc_out;
  inverter #(.w(6e-07)) Xinv1 (.vdd(?), .gnd(?), .inp(?), .out(?));
  inverter Xinv2 (.vdd(?), .gnd(?), .inp(?), .out(?));
  inverter Xinv3 (.vdd(?), .gnd(?), .inp(?), .out(?));
endmodule
module inverter (vdd, gnd, inp, out);
  inout vdd, gnd, inp, out;
  nfet #(.l(1e-07), .w(5e-07)) XMN (.p2(gnd), .p1(inp), .p0(out));
  pfet #(.l(1e-07), .w(1e-06)) XMP (.p2(vdd), .p1(inp), .p0(out));
endmodule

